# M1

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.utils import set_random_seed

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
set_random_seed(SEED)


# ============================================================
# 3. LOAD TRAIN / TEST DATA
# ============================================================

train_df = pd.read_csv(
    r"C:\\Users\\HPC\\Desktop\\DR.Sreelakshmi\\Dengue\\train_M1.csv"
)

test_df = pd.read_csv(
    r"C:\\Users\\HPC\\Desktop\\DR.Sreelakshmi\\Dengue\\test_M1.csv"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df.columns)
print(test_df.columns)


# ============================================================
# 4. COMBINE DATA
# ============================================================

# Keeping the same scaling approach as your original code
full_df = pd.concat(
    [train_df, test_df],
    axis=0
).reset_index(drop=True)


# ============================================================
# 5. SCALE DATA
# ============================================================

scaler = MinMaxScaler()

scaled_data = pd.DataFrame(
    scaler.fit_transform(full_df),
    columns=full_df.columns
)


# ============================================================
# 6. CREATE LSTM SEQUENCES
# ============================================================

def create_multivariate_dataset(
    dataset,
    target_index,
    look_back=1
):

    dataX = []
    dataY = []

    for i in range(
        len(dataset) - look_back
    ):

        a = dataset[
            i:(i + look_back),
            :
        ]

        dataX.append(a)

        dataY.append(
            dataset[
                i + look_back,
                target_index
            ]
        )

    return (
        np.array(dataX),
        np.array(dataY)
    )


# ============================================================
# 7. SELECT TARGET
# ============================================================

dataset = scaled_data.values.astype(
    'float32'
)

target_index = (
    scaled_data.columns.get_loc(
        "Dengue"
    )
)


# ============================================================
# 8. SELECTED LSTM M1 PARAMETERS
# ============================================================

LSTM_units = 128
Dense_units = 16
Activation = 'tanh'
Batch_size = 32
Look_back = 3
Epochs = 150


# ============================================================
# 9. CREATE SEQUENCES
# ============================================================

X, y = create_multivariate_dataset(
    dataset,
    target_index,
    Look_back
)


# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

train_size = (
    len(train_df) - Look_back
)

trainX = X[:train_size]
testX = X[train_size:]

trainY = y[:train_size]
testY = y[train_size:]


print("\nSequence shapes:")
print("Train X:", trainX.shape)
print("Train Y:", trainY.shape)
print("Test X :", testX.shape)
print("Test Y :", testY.shape)


# ============================================================
# 11. BUILD LSTM MODEL
# ============================================================

model = Sequential()

model.add(
    Input(
        shape=(
            Look_back,
            trainX.shape[2]
        )
    )
)

model.add(
    LSTM(
        LSTM_units,
        activation=Activation,
        return_sequences=True
    )
)

model.add(
    LSTM(
        LSTM_units,
        activation=Activation
    )
)

model.add(
    Dense(
        Dense_units,
        activation=Activation
    )
)

model.add(
    Dense(1)
)


# ============================================================
# 12. COMPILE MODEL
# ============================================================

model.compile(
    loss='mean_squared_error',
    optimizer='adam'
)


# ============================================================
# 13. TRAIN MODEL
# ============================================================

print("\n" + "=" * 60)
print("TRAINING DENGUE LSTM M1")
print("=" * 60)

print(f"LSTM units  : {LSTM_units}")
print(f"Dense units : {Dense_units}")
print(f"Activation  : {Activation}")
print(f"Batch size  : {Batch_size}")
print(f"Look-back   : {Look_back}")
print(f"Epochs      : {Epochs}")
print(f"Random seed : {SEED}")

history = model.fit(
    trainX,
    trainY,
    epochs=Epochs,
    batch_size=Batch_size,
    validation_data=(testX, testY),
    verbose=0,
    shuffle=False
)


# ============================================================
# 14. PREDICTIONS
# ============================================================

y_train_pred = model.predict(
    trainX,
    verbose=0
).flatten()

y_test_pred = model.predict(
    testX,
    verbose=0
).flatten()


# ============================================================
# 15. MAPE FUNCTION
# ============================================================

def calculate_mape(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove observed zero values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (
                y_true[mask]
                - y_pred[mask]
            )
            /
            y_true[mask]
        )
    ) * 100


# ============================================================
# 16. MPE FUNCTION
# ============================================================

def calculate_mpe(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove observed zero values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (
                y_true[mask]
                - y_pred[mask]
            )
            /
            y_true[mask]
        )
    ) * 100


# ============================================================
# 17. TRAIN METRICS
# ============================================================

train_rmse = np.sqrt(
    mean_squared_error(
        trainY,
        y_train_pred
    )
)

train_mae = mean_absolute_error(
    trainY,
    y_train_pred
)

train_mape = calculate_mape(
    trainY,
    y_train_pred
)

train_mpe = calculate_mpe(
    trainY,
    y_train_pred
)

train_r2 = r2_score(
    trainY,
    y_train_pred
)


# ============================================================
# 18. TEST METRICS
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(
        testY,
        y_test_pred
    )
)

test_mae = mean_absolute_error(
    testY,
    y_test_pred
)

test_mape = calculate_mape(
    testY,
    y_test_pred
)

test_mpe = calculate_mpe(
    testY,
    y_test_pred
)

test_r2 = r2_score(
    testY,
    y_test_pred
)


# ============================================================
# 19. PRINT PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("SELECTED DENGUE LSTM M1 PARAMETERS")
print("=" * 60)

print(f"LSTM units  : {LSTM_units}")
print(f"Dense units : {Dense_units}")
print(f"Activation  : {Activation}")
print(f"Batch size  : {Batch_size}")
print(f"Look-back   : {Look_back}")
print(f"Epochs      : {Epochs}")
print(f"Random seed : {SEED}")


# ============================================================
# 20. PRINT TRAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

print(f"RMSE : {train_rmse:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MAPE : {train_mape:.4f}%")
print(f"MPE  : {train_mpe:.4f}%")
print(f"R²   : {train_r2:.4f}")


# ============================================================
# 21. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MAPE : {test_mape:.4f}%")
print(f"MPE  : {test_mpe:.4f}%")
print(f"R²   : {test_r2:.4f}")


# ============================================================
# 22. COMPLETE
# ============================================================

print("\n" + "=" * 60)
print("LSTM DENGUE M1 MODEL RUN COMPLETED")
print("=" * 60)


Train shape: (4052, 32)
Test shape: (1014, 32)
Index(['Mean_temp', 'Max_temp', 'Min_temp', 'Diurnal_temp', 'Rainfall_mean',
       'Relative_humidity', 'Population_density', 'Mean_temp_lag1',
       'Mean_temp_lag2', 'Mean_temp_lag3', 'Max_temp_lag1', 'Max_temp_lag2',
       'Max_temp_lag3', 'Min_temp_lag1', 'Min_temp_lag2', 'Min_temp_lag3',
       'Diurnal_temp_lag1', 'Diurnal_temp_lag2', 'Diurnal_temp_lag3',
       'Rainfall_mean_lag1', 'Rainfall_mean_lag2', 'Rainfall_mean_lag3',
       'Relative_humidity_lag1', 'Relative_humidity_lag2',
       'Relative_humidity_lag3', 'Dengue_lag1', 'Season_Monsoon',
       'Season_Post_Monsoon', 'Season_Pre_Monsoon', 'Location_Rural',
       'Location_Urban', 'Dengue'],
      dtype='object')
Index(['Mean_temp', 'Max_temp', 'Min_temp', 'Diurnal_temp', 'Rainfall_mean',
       'Relative_humidity', 'Population_density', 'Mean_temp_lag1',
       'Mean_temp_lag2', 'Mean_temp_lag3', 'Max_temp_lag1', 'Max_temp_lag2',
       'Max_temp_lag3', 'Min_temp_lag

# M2

In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.utils import set_random_seed

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
set_random_seed(SEED)


# ============================================================
# 3. LOAD TRAIN / TEST DATA
# ============================================================

train_df = pd.read_csv(
    r"C:\\Users\\HPC\\Desktop\\DR.Sreelakshmi\\Dengue\\train_M2.csv"
)

test_df = pd.read_csv(
    r"C:\\Users\\HPC\\Desktop\\DR.Sreelakshmi\\Dengue\\test_M2.csv"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df.columns)
print(test_df.columns)


# ============================================================
# 4. COMBINE DATA
# ============================================================

# Keeping the same scaling approach as your original code
full_df = pd.concat(
    [train_df, test_df],
    axis=0
).reset_index(drop=True)


# ============================================================
# 5. SCALE DATA
# ============================================================

scaler = MinMaxScaler()

scaled_data = pd.DataFrame(
    scaler.fit_transform(full_df),
    columns=full_df.columns
)


# ============================================================
# 6. CREATE LSTM SEQUENCES
# ============================================================

def create_multivariate_dataset(
    dataset,
    target_index,
    look_back=1
):

    dataX = []
    dataY = []

    for i in range(
        len(dataset) - look_back
    ):

        a = dataset[
            i:(i + look_back),
            :
        ]

        dataX.append(a)

        dataY.append(
            dataset[
                i + look_back,
                target_index
            ]
        )

    return (
        np.array(dataX),
        np.array(dataY)
    )


# ============================================================
# 7. SELECT TARGET
# ============================================================

dataset = scaled_data.values.astype(
    'float32'
)

target_index = (
    scaled_data.columns.get_loc(
        "Dengue"
    )
)


# ============================================================
# 8. SELECTED LSTM M1 PARAMETERS
# ============================================================

LSTM_units = 32
Dense_units = 32
Activation = 'relu'
Batch_size = 32
Look_back = 6
Epochs = 100


# ============================================================
# 9. CREATE SEQUENCES
# ============================================================

X, y = create_multivariate_dataset(
    dataset,
    target_index,
    Look_back
)


# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

train_size = (
    len(train_df) - Look_back
)

trainX = X[:train_size]
testX = X[train_size:]

trainY = y[:train_size]
testY = y[train_size:]


print("\nSequence shapes:")
print("Train X:", trainX.shape)
print("Train Y:", trainY.shape)
print("Test X :", testX.shape)
print("Test Y :", testY.shape)


# ============================================================
# 11. BUILD LSTM MODEL
# ============================================================

model = Sequential()

model.add(
    Input(
        shape=(
            Look_back,
            trainX.shape[2]
        )
    )
)

model.add(
    LSTM(
        LSTM_units,
        activation=Activation,
        return_sequences=True
    )
)

model.add(
    LSTM(
        LSTM_units,
        activation=Activation
    )
)

model.add(
    Dense(
        Dense_units,
        activation=Activation
    )
)

model.add(
    Dense(1)
)


# ============================================================
# 12. COMPILE MODEL
# ============================================================

model.compile(
    loss='mean_squared_error',
    optimizer='adam'
)


# ============================================================
# 13. TRAIN MODEL
# ============================================================

print("\n" + "=" * 60)
print("TRAINING DENGUE LSTM M2")
print("=" * 60)

print(f"LSTM units  : {LSTM_units}")
print(f"Dense units : {Dense_units}")
print(f"Activation  : {Activation}")
print(f"Batch size  : {Batch_size}")
print(f"Look-back   : {Look_back}")
print(f"Epochs      : {Epochs}")
print(f"Random seed : {SEED}")

history = model.fit(
    trainX,
    trainY,
    epochs=Epochs,
    batch_size=Batch_size,
    validation_data=(testX, testY),
    verbose=0,
    shuffle=False
)


# ============================================================
# 14. PREDICTIONS
# ============================================================

y_train_pred = model.predict(
    trainX,
    verbose=0
).flatten()

y_test_pred = model.predict(
    testX,
    verbose=0
).flatten()


# ============================================================
# 15. MAPE FUNCTION
# ============================================================

def calculate_mape(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove observed zero values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (
                y_true[mask]
                - y_pred[mask]
            )
            /
            y_true[mask]
        )
    ) * 100


# ============================================================
# 16. MPE FUNCTION
# ============================================================

def calculate_mpe(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove observed zero values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (
                y_true[mask]
                - y_pred[mask]
            )
            /
            y_true[mask]
        )
    ) * 100


# ============================================================
# 17. TRAIN METRICS
# ============================================================

train_rmse = np.sqrt(
    mean_squared_error(
        trainY,
        y_train_pred
    )
)

train_mae = mean_absolute_error(
    trainY,
    y_train_pred
)

train_mape = calculate_mape(
    trainY,
    y_train_pred
)

train_mpe = calculate_mpe(
    trainY,
    y_train_pred
)

train_r2 = r2_score(
    trainY,
    y_train_pred
)


# ============================================================
# 18. TEST METRICS
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(
        testY,
        y_test_pred
    )
)

test_mae = mean_absolute_error(
    testY,
    y_test_pred
)

test_mape = calculate_mape(
    testY,
    y_test_pred
)

test_mpe = calculate_mpe(
    testY,
    y_test_pred
)

test_r2 = r2_score(
    testY,
    y_test_pred
)


# ============================================================
# 19. PRINT PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("SELECTED DENGUE LSTM M2 PARAMETERS")
print("=" * 60)

print(f"LSTM units  : {LSTM_units}")
print(f"Dense units : {Dense_units}")
print(f"Activation  : {Activation}")
print(f"Batch size  : {Batch_size}")
print(f"Look-back   : {Look_back}")
print(f"Epochs      : {Epochs}")
print(f"Random seed : {SEED}")


# ============================================================
# 20. PRINT TRAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

print(f"RMSE : {train_rmse:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MAPE : {train_mape:.4f}%")
print(f"MPE  : {train_mpe:.4f}%")
print(f"R²   : {train_r2:.4f}")


# ============================================================
# 21. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MAPE : {test_mape:.4f}%")
print(f"MPE  : {test_mpe:.4f}%")
print(f"R²   : {test_r2:.4f}")


# ============================================================
# 22. COMPLETE
# ============================================================

print("\n" + "=" * 60)
print("LSTM DENGUE M2 MODEL RUN COMPLETED")
print("=" * 60)

Train shape: (4052, 20)
Test shape: (1014, 20)
Index(['HI', 'BI', 'CI', 'HI_lag1', 'HI_lag2', 'HI_lag3', 'BI_lag1', 'BI_lag2',
       'BI_lag3', 'CI_lag1', 'CI_lag2', 'CI_lag3', 'Dengue_lag1',
       'Season_Monsoon', 'Season_Post_Monsoon', 'Season_Pre_Monsoon',
       'Location_Rural', 'Location_Urban', 'Population_density', 'Dengue'],
      dtype='object')
Index(['HI', 'BI', 'CI', 'HI_lag1', 'HI_lag2', 'HI_lag3', 'BI_lag1', 'BI_lag2',
       'BI_lag3', 'CI_lag1', 'CI_lag2', 'CI_lag3', 'Dengue_lag1',
       'Season_Monsoon', 'Season_Post_Monsoon', 'Season_Pre_Monsoon',
       'Location_Rural', 'Location_Urban', 'Population_density', 'Dengue'],
      dtype='object')

Sequence shapes:
Train X: (4046, 6, 20)
Train Y: (4046,)
Test X : (1014, 6, 20)
Test Y : (1014,)

TRAINING DENGUE LSTM M2
LSTM units  : 32
Dense units : 32
Activation  : relu
Batch size  : 32
Look-back   : 6
Epochs      : 100
Random seed : 42

SELECTED DENGUE LSTM M2 PARAMETERS
LSTM units  : 32
Dense units : 32
Activation 

# M3

In [3]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.utils import set_random_seed

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
set_random_seed(SEED)


# ============================================================
# 3. LOAD TRAIN / TEST DATA
# ============================================================

train_df = pd.read_csv(
    r"C:\\Users\\HPC\\Desktop\\DR.Sreelakshmi\\Dengue\\train_M3.csv"
)

test_df = pd.read_csv(
    r"C:\\Users\\HPC\\Desktop\\DR.Sreelakshmi\\Dengue\\test_M3.csv"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df.columns)
print(test_df.columns)


# ============================================================
# 4. COMBINE DATA
# ============================================================

# Keeping the same scaling approach as your original code
full_df = pd.concat(
    [train_df, test_df],
    axis=0
).reset_index(drop=True)


# ============================================================
# 5. SCALE DATA
# ============================================================

scaler = MinMaxScaler()

scaled_data = pd.DataFrame(
    scaler.fit_transform(full_df),
    columns=full_df.columns
)


# ============================================================
# 6. CREATE LSTM SEQUENCES
# ============================================================

def create_multivariate_dataset(
    dataset,
    target_index,
    look_back=1
):

    dataX = []
    dataY = []

    for i in range(
        len(dataset) - look_back
    ):

        a = dataset[
            i:(i + look_back),
            :
        ]

        dataX.append(a)

        dataY.append(
            dataset[
                i + look_back,
                target_index
            ]
        )

    return (
        np.array(dataX),
        np.array(dataY)
    )


# ============================================================
# 7. SELECT TARGET
# ============================================================

dataset = scaled_data.values.astype(
    'float32'
)

target_index = (
    scaled_data.columns.get_loc(
        "Dengue"
    )
)


# ============================================================
# 8. SELECTED LSTM M1 PARAMETERS
# ============================================================

LSTM_units = 32
Dense_units = 16
Activation = 'relu'
Batch_size = 32
Look_back = 6
Epochs = 150


# ============================================================
# 9. CREATE SEQUENCES
# ============================================================

X, y = create_multivariate_dataset(
    dataset,
    target_index,
    Look_back
)


# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

train_size = (
    len(train_df) - Look_back
)

trainX = X[:train_size]
testX = X[train_size:]

trainY = y[:train_size]
testY = y[train_size:]


print("\nSequence shapes:")
print("Train X:", trainX.shape)
print("Train Y:", trainY.shape)
print("Test X :", testX.shape)
print("Test Y :", testY.shape)


# ============================================================
# 11. BUILD LSTM MODEL
# ============================================================

model = Sequential()

model.add(
    Input(
        shape=(
            Look_back,
            trainX.shape[2]
        )
    )
)

model.add(
    LSTM(
        LSTM_units,
        activation=Activation,
        return_sequences=True
    )
)

model.add(
    LSTM(
        LSTM_units,
        activation=Activation
    )
)

model.add(
    Dense(
        Dense_units,
        activation=Activation
    )
)

model.add(
    Dense(1)
)


# ============================================================
# 12. COMPILE MODEL
# ============================================================

model.compile(
    loss='mean_squared_error',
    optimizer='adam'
)


# ============================================================
# 13. TRAIN MODEL
# ============================================================

print("\n" + "=" * 60)
print("TRAINING DENGUE LSTM M3")
print("=" * 60)

print(f"LSTM units  : {LSTM_units}")
print(f"Dense units : {Dense_units}")
print(f"Activation  : {Activation}")
print(f"Batch size  : {Batch_size}")
print(f"Look-back   : {Look_back}")
print(f"Epochs      : {Epochs}")
print(f"Random seed : {SEED}")

history = model.fit(
    trainX,
    trainY,
    epochs=Epochs,
    batch_size=Batch_size,
    validation_data=(testX, testY),
    verbose=0,
    shuffle=False
)


# ============================================================
# 14. PREDICTIONS
# ============================================================

y_train_pred = model.predict(
    trainX,
    verbose=0
).flatten()

y_test_pred = model.predict(
    testX,
    verbose=0
).flatten()


# ============================================================
# 15. MAPE FUNCTION
# ============================================================

def calculate_mape(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove observed zero values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (
                y_true[mask]
                - y_pred[mask]
            )
            /
            y_true[mask]
        )
    ) * 100


# ============================================================
# 16. MPE FUNCTION
# ============================================================

def calculate_mpe(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove observed zero values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (
                y_true[mask]
                - y_pred[mask]
            )
            /
            y_true[mask]
        )
    ) * 100


# ============================================================
# 17. TRAIN METRICS
# ============================================================

train_rmse = np.sqrt(
    mean_squared_error(
        trainY,
        y_train_pred
    )
)

train_mae = mean_absolute_error(
    trainY,
    y_train_pred
)

train_mape = calculate_mape(
    trainY,
    y_train_pred
)

train_mpe = calculate_mpe(
    trainY,
    y_train_pred
)

train_r2 = r2_score(
    trainY,
    y_train_pred
)


# ============================================================
# 18. TEST METRICS
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(
        testY,
        y_test_pred
    )
)

test_mae = mean_absolute_error(
    testY,
    y_test_pred
)

test_mape = calculate_mape(
    testY,
    y_test_pred
)

test_mpe = calculate_mpe(
    testY,
    y_test_pred
)

test_r2 = r2_score(
    testY,
    y_test_pred
)


# ============================================================
# 19. PRINT PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("SELECTED DENGUE LSTM M3 PARAMETERS")
print("=" * 60)

print(f"LSTM units  : {LSTM_units}")
print(f"Dense units : {Dense_units}")
print(f"Activation  : {Activation}")
print(f"Batch size  : {Batch_size}")
print(f"Look-back   : {Look_back}")
print(f"Epochs      : {Epochs}")
print(f"Random seed : {SEED}")


# ============================================================
# 20. PRINT TRAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

print(f"RMSE : {train_rmse:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MAPE : {train_mape:.4f}%")
print(f"MPE  : {train_mpe:.4f}%")
print(f"R²   : {train_r2:.4f}")


# ============================================================
# 21. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MAPE : {test_mape:.4f}%")
print(f"MPE  : {test_mpe:.4f}%")
print(f"R²   : {test_r2:.4f}")


# ============================================================
# 22. COMPLETE
# ============================================================

print("\n" + "=" * 60)
print("LSTM DENGUE M3 MODEL RUN COMPLETED")
print("=" * 60)

Train shape: (4052, 44)
Test shape: (1014, 44)
Index(['HI', 'BI', 'CI', 'Mean_temp', 'Max_temp', 'Min_temp', 'Diurnal_temp',
       'Rainfall_mean', 'Relative_humidity', 'Population_density',
       'Mean_temp_lag1', 'Mean_temp_lag2', 'Mean_temp_lag3', 'Max_temp_lag1',
       'Max_temp_lag2', 'Max_temp_lag3', 'Min_temp_lag1', 'Min_temp_lag2',
       'Min_temp_lag3', 'Diurnal_temp_lag1', 'Diurnal_temp_lag2',
       'Diurnal_temp_lag3', 'Rainfall_mean_lag1', 'Rainfall_mean_lag2',
       'Rainfall_mean_lag3', 'Relative_humidity_lag1',
       'Relative_humidity_lag2', 'Relative_humidity_lag3', 'HI_lag1',
       'HI_lag2', 'HI_lag3', 'BI_lag1', 'BI_lag2', 'BI_lag3', 'CI_lag1',
       'CI_lag2', 'CI_lag3', 'Dengue_lag1', 'Season_Monsoon',
       'Season_Post_Monsoon', 'Season_Pre_Monsoon', 'Location_Rural',
       'Location_Urban', 'Dengue'],
      dtype='object')
Index(['HI', 'BI', 'CI', 'Mean_temp', 'Max_temp', 'Min_temp', 'Diurnal_temp',
       'Rainfall_mean', 'Relative_humidity', 'Popul